# Returning to scatterplot and possibly some filtering for visibility analysis

Maybe a scatterplot is the most "honest" way to show the data? Also, I'm trying the same filtering process I did for the foliage just to test

In [31]:
# Imports
import rasterio
import rasterio.plot
from rasterio.windows import Window
import geopandas as gpd
from rasterio.mask import mask
import json
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from scipy.stats import pearsonr

import plotly.graph_objects as go

import cmcrameri.cm as cmc

import folium
from folium.plugins import HeatMap
import matplotlib.colors as mcolors
import matplotlib.cm as cm

import folium
from folium.plugins import HeatMap

import folium
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from scipy.interpolate import Rbf
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
import io
from PIL import Image

import pandas as pd

import folium
import numpy as np
from scipy.spatial import Voronoi
from shapely.geometry import Polygon
import geopandas as gpd
import json

In [2]:

park_vga = gpd.read_file("../../outputs/method_03_hex/Woodhouse_Moor_10_m_grid_hex_buffer_100m_vis_hex.geojson")
park_gdf= gpd.read_file("../../processed_datasets/Leeds_Woodhouse_Moor_park_boundaries.geojson")
park_vga.head()

TARGET_CRS = 'EPSG:27700'

# Ok, again I will have to convert the CRS
park_vga_bng = park_vga.to_crs(epsg=27700)

In [3]:
park_foliage = gpd.read_file("../../outputs/foliage_tests/Woodhouse_Moor_10_foliage_v1.geojson")

In [4]:
park_foliage.head()

,max_height,mean_height,std_height,count_pixels,percent_cover,percent_above_1m,percent_1_2m,percent_2_5m,percent_5_10m,percent_10m_plus,geometry
0,13.629997,4.036781,4.504664,73,0.016486,56.164384,6.849315,15.068493,17.808219,16.438356,POINT (429080.248 434705.602)
1,3.718994,3.718994,0.000000,1,0.000226,100.000000,0.000000,100.000000,0.000000,0.000000,POINT (429020.248 434722.922)
2,20.237999,2.453029,4.878776,69,0.015583,36.231884,14.492754,5.797101,4.347826,11.594203,POINT (429035.248 434722.922)
3,20.237999,2.713092,4.821730,153,0.034553,36.601307,4.575163,12.418301,9.150327,10.457516,POINT (429027.748 434731.582)
4,19.533997,3.193339,4.553010,257,0.058040,43.190661,3.501946,11.673152,16.731518,11.284047,POINT (429065.248 434740.243)


In [5]:
print(f"Original park CRS: {park_gdf.crs}")

# Reproject park to target CRS if needed
if park_gdf.crs != TARGET_CRS:
    park_gdf = park_gdf.to_crs(TARGET_CRS)
    print(f"Reprojected park to {TARGET_CRS}")
else:
    print(f"Park already in {TARGET_CRS}")

# Extract park boundary geometry (now in EPSG:27700)
park_boundary = park_gdf.geometry.iloc[0]
print(f"Park boundary CRS: {park_gdf.crs}")

Original park CRS: EPSG:4326
Reprojected park to EPSG:27700
Park boundary CRS: EPSG:27700


In [6]:
# Create interactive plotly map
fig = go.Figure()

# Extract coordinates for the scatter plot
lons = park_foliage.geometry.x
lats = park_foliage.geometry.y

# Create custom hover text
hover_text = [
    f"<b>Foliage Data</b><br>" +
    f"Mean Height: {mean:.2f}m<br>" +
    f"Vegetation >1m: {pct:.1f}%"
    for mean, pct in zip(
        park_foliage['mean_height'], 
        park_foliage['percent_above_1m']
    )
]


# Add scatter plot
fig.add_trace(go.Scatter(
    x=lons,
    y=lats,
    mode='markers',
    marker=dict(
        size=8,
        color=park_foliage['max_height'],
        colorscale='Greens',
        cmin=0,
        cmax=20,
        colorbar=dict(
            title="Max Height (m)",
            thickness=15,
            len=0.7
        ),
        line=dict(width=0.5, color='white'),
        opacity=0.8
    ),
    hovertext=hover_text,
    hoverinfo='text',
    name='Vegetation'
))

# Update layout for better visualization
fig.update_layout(
    title='Foliage Distribution - Interactive Map',
    xaxis_title='Easting (EPSG:27700)',
    yaxis_title='Northing (EPSG:27700)',
    hovermode='closest',
    height=700,
    width=1000,
    xaxis=dict(scaleanchor='y', scaleratio=1),
    yaxis=dict(scaleanchor='x', scaleratio=1),
    plot_bgcolor='rgba(240, 240, 240, 0.9)'
)

fig.show()

In [7]:
# Create interactive plotly map
fig = go.Figure()

# Extract coordinates for the scatter plot
lons = park_foliage.geometry.x
lats = park_foliage.geometry.y

# Create custom hover text
hover_text = [
    f"<b>Foliage Data</b><br>" +
    f"Mean Height: {mean:.2f}m<br>" +
    f"Vegetation >1m: {pct:.1f}%"
    for mean, pct in zip(
        park_foliage['mean_height'], 
        park_foliage['percent_above_1m']
    )
]


# Add scatter plot
fig.add_trace(go.Scatter(
    x=lons,
    y=lats,
    mode='markers',
    marker=dict(
        size=8,
        color=park_foliage['percent_above_1m'],
        colorscale='Greens',
        colorbar=dict(
            title="Foliage spatial coverage % (>1 m height)",
            thickness=15,
            len=0.7
        ),
        line=dict(width=0.5, color='white'),
        opacity=0.8
    ),
    hovertext=hover_text,
    hoverinfo='text',
    name='Vegetation'
))

# Update layout for better visualization
fig.update_layout(
    title='Foliage Distribution - Interactive Map',
    xaxis_title='Easting (EPSG:27700)',
    yaxis_title='Northing (EPSG:27700)',
    hovermode='closest',
    height=700,
    width=1000,
    xaxis=dict(scaleanchor='y', scaleratio=1),
    yaxis=dict(scaleanchor='x', scaleratio=1),
    plot_bgcolor='rgba(240, 240, 240, 0.9)'
)

fig.show()

In [8]:
# Create interactive plotly map
fig = go.Figure()

# Extract coordinates for the scatter plot
lons = park_vga_bng.geometry.x
lats = park_vga_bng.geometry.y


# Add scatter plot
fig.add_trace(go.Scatter(
    x=lons,
    y=lats,
    mode='markers',
    marker=dict(
        size=8,
        color=park_vga_bng['visibility_pct'],
        colorscale="Cividis",
        colorbar=dict(
            title="Visibility %",
            thickness=15,
            len=0.7
        ),
        line=dict(width=0.5, color='white'),
        opacity=0.8
    ),
    name='Vegetation'
))

# Update layout for better visualization
fig.update_layout(
    title='Visibility',
    xaxis_title='Easting (EPSG:27700)',
    yaxis_title='Northing (EPSG:27700)',
    hovermode='closest',
    height=700,
    width=1000,
    xaxis=dict(scaleanchor='y', scaleratio=1),
    yaxis=dict(scaleanchor='x', scaleratio=1),
    plot_bgcolor='rgba(240, 240, 240, 0.9)'
)

fig.show()

In [9]:
# Could potentially do some filtering like we have done with the foliage?
# foliage_stats_filtered = foliage_stats[(foliage_stats['percent_above_1m'] >= 30) & (foliage_stats['mean_height'] >= 1)].copy()

park_vga_bng.head()

,visible_points,total_points,visibility_pct,geometry
0,0,98,0.0,POINT (429065.248 434705.602)
1,0,96,0.0,POINT (429080.248 434705.602)
2,0,92,0.0,POINT (429095.248 434705.602)
3,0,86,0.0,POINT (429110.248 434705.602)
4,0,109,0.0,POINT (429042.748 434714.262)


In [10]:
vis_filtered = park_vga_bng[park_vga_bng['visibility_pct'] >= 15].copy()

In [11]:
# Create interactive plotly map
fig = go.Figure()

# Extract coordinates for the scatter plot
lons = vis_filtered.geometry.x
lats = vis_filtered.geometry.y


# Add scatter plot
fig.add_trace(go.Scatter(
    x=lons,
    y=lats,
    mode='markers',
    marker=dict(
        size=8,
        color=vis_filtered['visibility_pct'],
        colorscale="Cividis",
        colorbar=dict(
            title="Visibility %",
            thickness=15,
            len=0.7
        ),
        line=dict(width=0.5, color='white'),
        opacity=0.8
    ),
    name='Vegetation'
))

# Update layout for better visualization
fig.update_layout(
    title='Visibility',
    xaxis_title='Easting (EPSG:27700)',
    yaxis_title='Northing (EPSG:27700)',
    hovermode='closest',
    height=700,
    width=1000,
    xaxis=dict(scaleanchor='y', scaleratio=1),
    yaxis=dict(scaleanchor='x', scaleratio=1),
    plot_bgcolor='rgba(240, 240, 240, 0.9)'
)

fig.show()

In [12]:
# Create interactive plotly map with toggleable layers
fig = go.Figure()

# Extract coordinates for foliage data
foliage_lons = park_foliage.geometry.x
foliage_lats = park_foliage.geometry.y

# Create hover text for foliage
foliage_hover_text = [
    f"<b>Foliage Data</b><br>" +
    f"Mean Height: {mean:.2f}m<br>" +
    f"Vegetation >1m: {pct:.1f}%"
    for mean, pct in zip(
        park_foliage['mean_height'], 
        park_foliage['percent_above_1m']
    )
]

# Add foliage trace
fig.add_trace(go.Scatter(
    x=foliage_lons,
    y=foliage_lats,
    mode='markers',
    marker=dict(
        size=8,
        color=park_foliage['percent_above_1m'],
        colorscale='Greens',
        colorbar=dict(
            title="Foliage %",
            thickness=15,
            len=0.7,
            x=1.02
        ),
        line=dict(width=0.5, color='white'),
        opacity=0.8
    ),
    hovertext=foliage_hover_text,
    hoverinfo='text',
    name='Foliage (Filtered)',
    visible=True
))

# Extract coordinates for visibility data
vis_lons = vis_filtered.geometry.x
vis_lats = vis_filtered.geometry.y

# Create hover text for vis
vis_hover_text = [
    f"<b>Estimated visibility</b><br>" +
    f"Visibility: {pct:.1f}%"
    for pct in vis_filtered['visibility_pct']
]

# Add visibility trace
fig.add_trace(go.Scatter(
    x=vis_lons,
    y=vis_lats,
    mode='markers',
    marker=dict(
        size=8,
        color=vis_filtered['visibility_pct'],
        colorscale='Cividis',
        colorbar=dict(
            title="Visibility %",
            thickness=15,
            len=0.7,
            x=1.15
        ),
        line=dict(width=0.5, color='white'),
        opacity=0.8
    ),
    hovertext=vis_hover_text,
    hoverinfo='text',
    name='Visibility (Filtered)',
    visible=True
))

x_coords = list(park_boundary.exterior.xy[0])
y_coords = list(park_boundary.exterior.xy[1])


fig.add_trace(go.Scatter(
    x=x_coords,
    y=y_coords,
    mode='lines',
    name='Park Boundary',
    line=dict(color='black', width=2),
    hoverinfo='skip'
))


# Update layout for better visualization
fig.update_layout(
    title='Foliage & Visibility Analysis - Interactive Map',
    xaxis_title='Easting (EPSG:27700)',
    yaxis_title='Northing (EPSG:27700)',
    hovermode='closest',
    height=700,
    width=1000,
    xaxis=dict(scaleanchor='y', scaleratio=1),
    yaxis=dict(scaleanchor='x', scaleratio=1),
    plot_bgcolor='rgba(240, 240, 240, 0.9)'
)



fig.show()

Now looking at folium, as this uses leaflet and is closer to our actual dashboard technology stack

In [13]:


# Convert data to WGS84 (lat/lon) for folium
park_foliage_wgs84 = park_foliage.to_crs(epsg=4326)
vis_filtered_wgs84 = vis_filtered.to_crs(epsg=4326)
park_boundary_wgs84 = park_gdf.to_crs(epsg=4326)
park_boundary = park_boundary_wgs84.geometry.iloc[0]

# Create base map centered on park
park_center = [
    park_foliage_wgs84.geometry.y.mean(),
    park_foliage_wgs84.geometry.x.mean()
]

m = folium.Map(
    location=park_center,
    zoom_start=15,
    tiles='cartodb positron',
)

# Create foliage heatmap layer
foliage_heat_data = [
    [point.y, point.x, val] 
    for point, val in zip(park_foliage_wgs84.geometry, park_foliage_wgs84['percent_above_1m'])
]

foliage_layer = folium.FeatureGroup(name='Foliage Heatmap')
HeatMap(foliage_heat_data, radius=15, blur=25, max_zoom=1).add_to(foliage_layer)
foliage_layer.add_to(m)

# Create visibility heatmap layer
vis_heat_data = [
    [point.y, point.x, val] 
    for point, val in zip(vis_filtered_wgs84.geometry, vis_filtered_wgs84['visibility_pct'])
]

vis_layer = folium.FeatureGroup(name='Visibility Heatmap')
HeatMap(vis_heat_data, radius=15, blur=25, max_zoom=1).add_to(vis_layer)
vis_layer.add_to(m)

# Add park boundary
coords = [(y, x) for x, y in zip(park_boundary.exterior.xy[0], park_boundary.exterior.xy[1])]
folium.PolyLine(
    coords,
    color='black',
    weight=2,
    opacity=0.8
).add_to(m)

# Add layer control
folium.LayerControl().add_to(m)

m.show_in_browser()

Your map should have been opened in your browser automatically.
Press ctrl+c to return.


In [ ]:


# Convert data to WGS84
park_foliage_wgs84 = park_foliage.to_crs(epsg=4326)
vis_filtered_wgs84 = vis_filtered.to_crs(epsg=4326)
park_boundary_wgs84 = park_gdf.to_crs(epsg=4326)
park_boundary = park_boundary_wgs84.geometry.iloc[0]

# Create base map
park_center = [
    park_foliage_wgs84.geometry.y.mean(),
    park_foliage_wgs84.geometry.x.mean()
]

m = folium.Map(
    location=park_center,
    zoom_start=15,
    tiles='cartodb positron',
)

# Function to add value-based heatmap
def add_value_heatmap(gdf, values, colormap_name, layer_name, label_name):
    cmap = cm.get_cmap(colormap_name)
    norm = mcolors.Normalize(vmin=values.min(), vmax=values.max())
    
    fg = folium.FeatureGroup(name=layer_name)
    
    for point, value in zip(gdf.geometry, values):
        color = mcolors.rgb2hex(cmap(norm(value)))
        folium.CircleMarker(
            location=[point.y, point.x],
            radius=5,
            color=color,
            fill=True,
            fillColor=color,
            fillOpacity=0.7,
            weight=0,
            popup=f'{label_name}: {value:.1f}%'
        ).add_to(fg)
    
    fg.add_to(m)

# Add foliage layer
add_value_heatmap(
    park_foliage_wgs84,
    park_foliage_wgs84['percent_above_1m'],
    'Greens',
    'Foliage Heatmap',
    'Foliage'
)

# Add visibility layer
add_value_heatmap(
    vis_filtered_wgs84,
    vis_filtered_wgs84['visibility_pct'],
    cmc.nuuk,
    'Visibility Heatmap',
    'Visibility'
)

# Add park boundary
coords = [(y, x) for x, y in zip(park_boundary.exterior.xy[0], park_boundary.exterior.xy[1])]
folium.PolyLine(
    coords,
    color='black',
    weight=2,
    opacity=0.8
).add_to(m)

folium.LayerControl().add_to(m)
m.show_in_browser()

In [32]:


# Convert to WGS84
park_vga_wgs84 = park_vga_bng.to_crs(epsg=4326)
park_boundary_wgs84 = park_gdf.to_crs(epsg=4326)
park_boundary = park_boundary_wgs84.geometry.iloc[0]

# Extract coordinates and values
coords = np.array([[pt.x, pt.y] for pt in park_vga_wgs84.geometry])
values = park_vga_wgs84['visibility_pct'].values

# Create Voronoi diagram
vor = Voronoi(coords)

# Create polygons from Voronoi cells
polygons = []
for point_index, region_index in enumerate(vor.point_region):
    region = vor.regions[region_index]
    if -1 not in region and len(region) > 0:  # Exclude infinite regions
        try:
            polygon = Polygon([vor.vertices[i] for i in region])
            if polygon.is_valid:
                polygons.append({
                    'geometry': polygon,
                    'visibility_pct': values[point_index]
                })
        except:
            pass

# Create GeoDataFrame
voronoi_gdf = gpd.GeoDataFrame(polygons, crs='EPSG:4326')

# Create base map
park_center = [park_vga_wgs84.geometry.y.mean(), park_vga_wgs84.geometry.x.mean()]

m = folium.Map(
    location=park_center,
    zoom_start=15,
    tiles='cartodb positron'
)

# Create choropleth from Voronoi
vmin = voronoi_gdf['visibility_pct'].min()
vmax = voronoi_gdf['visibility_pct'].max()

geojson_data = json.loads(voronoi_gdf.to_json())

def style_function(feature):
    value = feature['properties']['visibility_pct']
    norm = (value - vmin) / (vmax - vmin) if vmax > vmin else 0.5
    
    cmap = cmc.nuuk
    color = mcolors.rgb2hex(cmap(norm))
    
    return {
        'fillColor': color,
        'color': 'black',
        'weight': 0.5,
        'fillOpacity': 0.8
    }

folium.GeoJson(
    data=geojson_data,
    style_function=style_function,
    popup=folium.GeoJsonPopup(fields=['visibility_pct'])
).add_to(m)

# Add park boundary
coords_boundary = [(y, x) for x, y in zip(park_boundary.exterior.xy[0], park_boundary.exterior.xy[1])]
folium.PolyLine(coords_boundary, color='black', weight=2, opacity=0.8).add_to(m)

m.show_in_browser()

765:773: execution error: Safari got an error: Application isn’t running. (-600)


Your map should have been opened in your browser automatically.
Press ctrl+c to return.


In [42]:
# Basic working hex cholorpleth

# Convert to WGS84
park_vga_wgs84 = park_vga_bng.to_crs(epsg=4326)
park_boundary_wgs84 = park_gdf.to_crs(epsg=4326)
park_boundary = park_boundary_wgs84.geometry.iloc[0]

# Extract coordinates and values
coords = np.array([[pt.x, pt.y] for pt in park_vga_wgs84.geometry])
values = park_vga_wgs84['visibility_pct'].values

# Create Voronoi diagram
vor = Voronoi(coords)

# Create polygons from Voronoi cells
polygons = []
for point_index, region_index in enumerate(vor.point_region):
    region = vor.regions[region_index]
    if -1 not in region and len(region) > 0:
        try:
            polygon = Polygon([vor.vertices[i] for i in region])
            if polygon.is_valid:
                # Clip polygon to park boundary
                clipped = polygon.intersection(park_boundary)
                if clipped.area > 0:  # Only include if there's overlap
                    polygons.append({
                        'geometry': clipped,
                        'visibility_pct': values[point_index]
                    })
        except:
            pass

# Create GeoDataFrame
voronoi_gdf = gpd.GeoDataFrame(polygons, crs='EPSG:4326')

# Create base map
park_center = [park_vga_wgs84.geometry.y.mean(), park_vga_wgs84.geometry.x.mean()]

m = folium.Map(
    location=park_center,
    zoom_start=15,
    tiles='cartodb positron'
)

# Create choropleth from Voronoi
vmin = voronoi_gdf['visibility_pct'].min()
vmax = voronoi_gdf['visibility_pct'].max()

geojson_data = json.loads(voronoi_gdf.to_json())

def style_function(feature):
    value = feature['properties']['visibility_pct']
    norm = (value - vmin) / (vmax - vmin) if vmax > vmin else 0.5
    
    cmap = cmc.nuuk
    color = mcolors.rgb2hex(cmap(norm))
    
    return {
        'fillColor': color,
        'color': 'black',
        'weight': 0.5,
        'fillOpacity': 0.8
    }

folium.GeoJson(
    data=geojson_data,
    style_function=style_function,
    popup=folium.GeoJsonPopup(fields=['visibility_pct'])
).add_to(m)

# Add park boundary
coords_boundary = [(y, x) for x, y in zip(park_boundary.exterior.xy[0], park_boundary.exterior.xy[1])]
folium.PolyLine(coords_boundary, color='red', weight=2, opacity=0.8).add_to(m)

m.show_in_browser()

765:773: execution error: Safari got an error: Application isn’t running. (-600)


Your map should have been opened in your browser automatically.
Press ctrl+c to return.


In [44]:
voronoi_gdf.head()

,geometry,visibility_pct
0,"POLYGON ((-1.55992 53.80809, -1.55986 53.80808...",6.722689
1,"POLYGON ((-1.55969 53.80808, -1.55963 53.80808...",7.017544
2,"POLYGON ((-1.56026 53.80816, -1.5602 53.80816,...",5.882353
3,"POLYGON ((-1.55997 53.80816, -1.55992 53.80809...",5.263158
4,"POLYGON ((-1.55986 53.80808, -1.5598 53.80816,...",3.100775


In [39]:
# Convert to WGS84
park_vga_wgs84 = park_vga_bng.to_crs(epsg=4326)
park_foliage_wgs84 = park_foliage.to_crs(epsg=4326)
park_boundary_wgs84 = park_gdf.to_crs(epsg=4326)
park_boundary = park_boundary_wgs84.geometry.iloc[0]

# Extract coordinates and values
coords = np.array([[pt.x, pt.y] for pt in park_vga_wgs84.geometry])
values = park_vga_wgs84['visibility_pct'].values

# Create Voronoi diagram
vor = Voronoi(coords)

# Create polygons from Voronoi cells
polygons = []
for point_index, region_index in enumerate(vor.point_region):
    region = vor.regions[region_index]
    if -1 not in region and len(region) > 0:
        try:
            polygon = Polygon([vor.vertices[i] for i in region])
            if polygon.is_valid:
                clipped = polygon.intersection(park_boundary)
                if clipped.area > 0:
                    polygons.append({
                        'geometry': clipped,
                        'visibility_pct': values[point_index]
                    })
        except:
            pass

# Create GeoDataFrame
voronoi_gdf = gpd.GeoDataFrame(polygons, crs='EPSG:4326')

# Create base map
park_center = [park_vga_wgs84.geometry.y.mean(), park_vga_wgs84.geometry.x.mean()]

m = folium.Map(
    location=park_center,
    zoom_start=15,
    tiles='cartodb positron'
)

# Visibility layer
vmin = voronoi_gdf['visibility_pct'].min()
vmax = voronoi_gdf['visibility_pct'].max()

geojson_data = json.loads(voronoi_gdf.to_json())

def style_function(feature):
    value = feature['properties']['visibility_pct']
    
    # Transparent if under 15%
    if value < 15:
        return {
            'fillColor': '#ffffff',
            'color': '#ffffff',
            'weight': 0,
            'fillOpacity': 0
        }
    
    norm = (value - vmin) / (vmax - vmin) if vmax > vmin else 0.5
    cmap = cmc.nuuk
    color = mcolors.rgb2hex(cmap(norm))
    
    return {
        'fillColor': color,
        'color': 'black',
        'weight': 0.5,
        'fillOpacity': 0.8
    }

vis_layer = folium.FeatureGroup(name='Visibility Choropleth')
folium.GeoJson(
    data=geojson_data,
    style_function=style_function,
    popup=folium.GeoJsonPopup(fields=['visibility_pct'])
).add_to(vis_layer)
vis_layer.add_to(m)

# Foliage points layer
foliage_layer = folium.FeatureGroup(name='Foliage Points')

cmap_foliage = cm.get_cmap('Greens')
norm_foliage = mcolors.Normalize(
    vmin=park_foliage_wgs84['percent_above_1m'].min(),
    vmax=park_foliage_wgs84['percent_above_1m'].max()
)

for point, value in zip(park_foliage_wgs84.geometry, park_foliage_wgs84['percent_above_1m']):
    color = mcolors.rgb2hex(cmap_foliage(norm_foliage(value)))
    folium.CircleMarker(
        location=[point.y, point.x],
        radius=4,
        color=color,
        fill=True,
        fillColor=color,
        fillOpacity=0.8,
        weight=0.5,
        popup=f'Foliage: {value:.1f}%'
    ).add_to(foliage_layer)

foliage_layer.add_to(m)

# Add park boundary
coords_boundary = [(y, x) for x, y in zip(park_boundary.exterior.xy[0], park_boundary.exterior.xy[1])]
folium.PolyLine(coords_boundary, color='red', weight=2, opacity=0.8).add_to(m)

# Add layer control
folium.LayerControl().add_to(m)

m.show_in_browser()

/var/folders/3s/z86xkbn12m94tfxyzcgrwf080000gn/T/ipykernel_5696/1376217635.py:83: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap_foliage = cm.get_cmap('Greens')
765:773: execution error: Safari got an error: Application isn’t running. (-600)


Your map should have been opened in your browser automatically.
Press ctrl+c to return.


Trying out these different approaches with the tesselation and the voronoi interpolations

Ok that absolutely doesn't work! 

Options:

- Save the full dataset, do voronoi tesselation and plot it as a hex layer like that
    - Can create transparent sections if required/cut the hexes?
- Partial datasets, can plot as point data

Will need to save out the full foliage point dataset and then tesselate; then can filter

Will use something like this on the dashboard:

```
// Load GeoJSON
fetch('voronoi_visibility.geojson')
  .then(res => res.json())
  .then(data => {
    L.geoJSON(data, {
      style: feature => ({
        fillColor: feature.properties.visibility_pct > 15 ? 
          getColor(feature.properties.visibility_pct) : 'transparent',
        fillOpacity: 0.8,
        weight: 0.5
      })
    }).addTo(map);
  });
```

In [47]:
# filtering after the voronoi tesselation

# Convert to WGS84
park_vga_wgs84 = park_vga_bng.to_crs(epsg=4326)
park_boundary_wgs84 = park_gdf.to_crs(epsg=4326)
park_boundary = park_boundary_wgs84.geometry.iloc[0]

# Extract coordinates and values
coords = np.array([[pt.x, pt.y] for pt in park_vga_wgs84.geometry])
values = park_vga_wgs84['visibility_pct'].values

# Create Voronoi diagram
vor = Voronoi(coords)

# Create polygons from Voronoi cells
polygons = []
for point_index, region_index in enumerate(vor.point_region):
    region = vor.regions[region_index]
    if -1 not in region and len(region) > 0:
        try:
            polygon = Polygon([vor.vertices[i] for i in region])
            if polygon.is_valid:
                # Clip polygon to park boundary
                clipped = polygon.intersection(park_boundary)
                if clipped.area > 0:  # Only include if there's overlap
                    polygons.append({
                        'geometry': clipped,
                        'visibility_pct': values[point_index]
                    })
        except:
            pass

# Create GeoDataFrame
voronoi_gdf = gpd.GeoDataFrame(polygons, crs='EPSG:4326')

# but then filter it!
voronoi_gdf = voronoi_gdf[voronoi_gdf['visibility_pct'] >= 10].copy()

# Create base map
park_center = [park_vga_wgs84.geometry.y.mean(), park_vga_wgs84.geometry.x.mean()]

m = folium.Map(
    location=park_center,
    zoom_start=15,
    tiles='cartodb positron'
)

# Create choropleth from Voronoi
vmin = voronoi_gdf['visibility_pct'].min()
vmax = voronoi_gdf['visibility_pct'].max()

geojson_data = json.loads(voronoi_gdf.to_json())

def style_function(feature):
    value = feature['properties']['visibility_pct']
    norm = (value - vmin) / (vmax - vmin) if vmax > vmin else 0.5
    
    cmap = cmc.nuuk
    color = mcolors.rgb2hex(cmap(norm))
    
    return {
        'fillColor': color,
        'color': 'black',
        'weight': 0.5,
        'fillOpacity': 0.8
    }

folium.GeoJson(
    data=geojson_data,
    style_function=style_function,
    popup=folium.GeoJsonPopup(fields=['visibility_pct'])
).add_to(m)

# Add park boundary
coords_boundary = [(y, x) for x, y in zip(park_boundary.exterior.xy[0], park_boundary.exterior.xy[1])]
folium.PolyLine(coords_boundary, color='red', weight=2, opacity=0.8).add_to(m)

m.show_in_browser()

765:773: execution error: Safari got an error: Application isn’t running. (-600)


Your map should have been opened in your browser automatically.
Press ctrl+c to return.
